In [ ]:
%pip install polars

In [263]:
import polars as pl
import os
from dataclasses import dataclass, field
from typing import Optional
import altair as alt
alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

In [264]:
os.chdir("/home/tsetsi/Python/financial-data-science-jupyter")
WORK_DIR = os.getcwd()
DATA = os.path.join(WORK_DIR, "data")

In [265]:
files = {
    "quotes_inc_eu": "DE0007500001_quotes_incremental.csv",
    "quotes_inc_us": "US2561631068_quotes_incremental.csv",
    "trades_eu": "DE0007500001_trades.csv",
    "trades_us": "US2561631068_trades.csv"
}

In [266]:
quotes_inc_eu_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns"),
    "price": pl.Float64,
    "best_bid_price": pl.Float64,
    "best_ask_price": pl.Float64,
}

trades_eu_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns")
}

quotes_inc_us_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns")
}

trades_us_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns")
}

In [267]:
@dataclass
class LimitOrderBook:
    file: str
    folder_path: str
    df: pl.LazyFrame = field(init=False)
    schema_override: Optional[dict] = None
    separator: str = ","

    def __post_init__(self):
        # pl.scan_csv doesn't load the file into memory immediately but only when called with the
        # .collect() method, which generally makes running the code significantly faster.
        # schema_overrides is optional and can be used to explicitly set a data type to a column,
        # but it will return an error if polars finds some kind of mismatch.
        # def get_data(file: str, separator: str = ",", schema_overrides: dict = None) -> pl.LazyFrame:
        self.df = pl.scan_csv(
            source=f"{self.folder_path}/{self.file}",
            separator=self.separator,
            schema_overrides=self.schema_override
        )

In [268]:
## EU: Incremental quotes
quotes_inc_eu = LimitOrderBook(
    file=files["quotes_inc_eu"],
    folder_path=DATA,
    schema_override=quotes_inc_eu_schema
)

In [269]:
quotes_inc_eu = (
    quotes_inc_eu.df.sort(by=[pl.col("original_order_id"), 
                              pl.col("event_timestamp")]
))


In [270]:
# Sanity check: No original_order_id's exist within more than one venue.
print(
    quotes_inc_eu.group_by("original_order_id") \
        .agg([
            pl.col("venue").unique().alias("venues")
        ])
        .filter(pl.col("venues").list.len() > 1)
        .collect()
)

shape: (0, 2)
┌───────────────────┬───────────┐
│ original_order_id ┆ venues    │
│ ---               ┆ ---       │
│ i64               ┆ list[str] │
╞═══════════════════╪═══════════╡
└───────────────────┴───────────┘


In [271]:
def show_histogram(df: pl.LazyFrame, column_name: str, x_label: str = None, y_label: str = None) -> None:
    
    x=alt.X(f"{column_name}:N", sort="-y", axis=alt.Axis(labelAngle=0, title=x_label or column_name))
    
    (
        alt.layer(
            alt.Chart(df.collect()).mark_bar().encode(
                x=x,
                y="count():Q",
            ),
            alt.Chart(df.collect()).mark_text(dy=-5).encode(
                x=x,
                y="count():Q",
                text="count():Q",
            ),
        )
        .properties(width=750, height=250)
    ).display()

In [272]:
show_histogram(
    df=quotes_inc_eu, 
    column_name="venue",
    x_label="Venues"
)
show_histogram(
    df=quotes_inc_eu, 
    column_name="market_state",
    x_label="Market states"
)
show_histogram(
    df=quotes_inc_eu, 
    column_name="lob_action",
    x_label="LOB Actions"
)

alt.LayerChart(...)

alt.LayerChart(...)

alt.LayerChart(...)

In [273]:
# Sanity check: Every order should have a REMOVE operation at some point
# and at some market state
orders_without_remove = (
    quotes_inc_eu.group_by(pl.col("original_order_id"))
    .agg(pl.col("lob_action").unique().alias("lob_actions"))
    .filter(
        # pl.col("lob_actions").list.contains("INSERT") &
        pl.col("lob_actions").list.contains("REMOVE").not_()
    )
)

orders_without_remove.collect().show(limit=20)

original_order_id,lob_actions
i64,list[str]


# 3.1.

In [274]:
quotes_aggs = [
    pl.col("venue") \
        .first()
        .alias("venue"),
    pl.col("event_timestamp") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("insertion_date"),
    pl.col("event_timestamp") \
        .filter(pl.col("lob_action")=="REMOVE")
        .first()
        .alias("removal_date"),
    pl.col("event_timestamp") \
        .max()
        .alias("latest_event_timestamp"),
    pl.col("lob_action") \
        .eq("UPDATE")
        .sum()
        .alias("number_of_updates"),
    pl.col("price") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("price_at_insertion"),
    pl.col("size") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("size_at_insertion"),
    # EXECUTION SIZE
    # This sums the execution sizes of all original_order_id's
    # in preparation for determining the removal mechanism.
    pl.col("execution_size") \
        .filter(pl.col("order_executed")==True)
        .sum() # .first()
        .alias("execution_size"),
    pl.col("price_level") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("insertion_level"),
    pl.col("best_bid_price") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("best_bid_price_at_insertion"),
    pl.col("best_ask_price") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("best_ask_price_at_insertion"),
    pl.struct(
        [
            pl.col("event_timestamp") \
                .filter(pl.col("lob_action")=="UPDATE")
                .alias("lob_updates"),
            pl.col("price") \
                .filter(pl.col("lob_action")=="UPDATE")
                .alias("updated_prices"),
            pl.col("size") \
                .filter(pl.col("lob_action")=="UPDATE")
                .alias("updated_sizes"),
        ]),
    pl.col("market_state") \
        .filter(pl.col("lob_action")=="REMOVE")
        .first()
        .alias("market_state_at_removal"),
]

quotes_calc = [
    (
        pl.when(pl.col("removal_date").is_not_null())
            .then(pl.col("removal_date"))
            .otherwise(pl.col("latest_event_timestamp"))
        - pl.col("insertion_date")
    ).alias("order_lifetime"),

    # (best bid price + best ask price) / 2
    (
        (
            pl.col("best_bid_price_at_insertion")
            + pl.col("best_ask_price_at_insertion")
        ) / 2
    ).alias("midpoint_at_insertion"),
]

In [275]:
quotes_collapsed = (
    quotes_inc_eu
    .group_by("original_order_id")
    .agg(quotes_aggs)
    .with_columns(quotes_calc)
)

quotes_collapsed = quotes_collapsed.with_columns([
    # Absolute distance to midpoint:
    # |Price at insertion - Midpoint at Insertion|
    (
        pl.col("price_at_insertion")
        .sub(pl.col("midpoint_at_insertion"))
        .abs()
        .alias("abs_distance_to_midpoint")
    ),
    # Relative distance to midpoint:
    # |Price at insertion - Midpoint at Insertion|
    # / Midpoint at Insertion
    (
        (
            pl.col("price_at_insertion")
            .sub(pl.col("midpoint_at_insertion"))
            .abs()
        )
        .truediv(pl.col("midpoint_at_insertion"))
        .alias("rel_distance_to_midpoint")
    ),
    # Log order lifetime
    # All order lifetimes need to be > 0, otherwise this would throw an error.
    (
        pl.col("order_lifetime")
            .log()
            .alias("log_order_lifetime")
    ),
    # Removal mechanism:
    # Given the cumulative execution size over the entire order lifecycle, I assume:
    # 1. If execution_size = 0, then nothing has been traded and the order has been cancelled.
    # 2. If 0 > execution_size > size_at_insertion, then the order has been filled partially
    # and then cancelled (removed).
    # 3. If execution_size = size_at_insertion, then the entire order has been filled.
    (
        pl.when(pl.col("execution_size") == 0)
            .then(pl.lit("Cancel"))
            .when(pl.col("execution_size") < pl.col("size_at_insertion"))
            .then(pl.lit("Trade (partial)"))
            .when(pl.col("execution_size") == pl.col("size_at_insertion"))
            .then(pl.lit("Trade (full)"))
            .otherwise(pl.lit("Unknown"))
        .alias("removal_mechanism")
    ),
])

In [276]:
# Sanity check: No order should have an order lifetime <= 0,
# which would mess up the logarithm.
quotes_collapsed \
    .filter(pl.col("order_lifetime") <= 0) \
    .collect() \
    .show(limit=10)

original_order_id,venue,insertion_date,removal_date,latest_event_timestamp,number_of_updates,price_at_insertion,size_at_insertion,execution_size,insertion_level,best_bid_price_at_insertion,best_ask_price_at_insertion,lob_updates,market_state_at_removal,order_lifetime,midpoint_at_insertion,abs_distance_to_midpoint,rel_distance_to_midpoint,log_order_lifetime,removal_mechanism
i64,str,datetime[ns],datetime[ns],datetime[ns],u32,f64,i64,i64,i64,f64,f64,list[struct[3]],str,duration[ns],f64,f64,f64,f64,str


# 3.2

## Testing area


In [277]:
quotes_inc_eu \
    .filter((pl.col("order_executed")==False)) \
    .collect() \
    .show(limit=10)

side,price,size,order_id,event_timestamp,lob_action,old_price,old_size,old_order_id,order_executed,execution_price,execution_size,price_level,old_price_level,best_ask_price,best_ask_size,total_ask_size,best_bid_price,best_bid_size,total_bid_size,is_new_best_price,is_new_best_size,original_order_id,trade_id,size_ahead,orders_ahead,best_ask_num_orders,best_bid_num_orders,level_num_orders_total,level_size_total,total_ask_orders,total_bid_orders,market_state,venue
str,f64,i64,i64,datetime[ns],str,f64,i64,i64,bool,f64,i64,i64,i64,f64,i64,i64,f64,i64,i64,bool,bool,i64,i128,i64,i64,i64,i64,i64,i64,i64,i64,str,str
"""ASK""",8.1,947,50137,2023-09-01 07:00:23.832485,"""INSERT""",null,0,null,false,null,0,1,0,8.1,947,947,null,0,0,true,true,50137,0,0,0,1,0,1,947,1,0,"""CONTINUOUS_TRADING""","""AQEU"""
"""ASK""",null,0,null,2023-09-01 11:00:00.100847,"""REMOVE""",8.1,947,50137,false,null,0,0,10,7.304,750,6691,7.272,596,3742,false,false,50137,0,6691,11,1,1,0,0,11,7,"""CONTINUOUS_TRADING""","""AQEU"""
"""BID""",5.0,757,50138,2023-09-01 07:00:23.832985,"""INSERT""",null,0,null,false,null,0,1,0,8.1,947,947,5.0,757,757,true,true,50138,0,0,0,1,1,1,757,1,1,"""CONTINUOUS_TRADING""","""AQEU"""
"""BID""",null,0,null,2023-09-01 11:00:00.100871,"""REMOVE""",5.0,757,50138,false,null,0,0,7,7.304,750,6691,7.272,596,2985,false,false,50138,0,2985,6,1,1,0,0,11,6,"""CONTINUOUS_TRADING""","""AQEU"""
"""BID""",7.104,750,51441,2023-09-01 07:00:24.633269,"""INSERT""",null,0,null,false,null,0,1,0,8.1,947,947,7.104,750,1507,true,true,51441,0,0,0,1,1,1,750,1,2,"""CONTINUOUS_TRADING""","""AQEU"""
"""BID""",null,0,null,2023-09-01 07:00:24.642583,"""REMOVE""",7.104,750,51441,false,null,0,1,1,8.1,947,947,7.104,750,1507,false,true,51441,0,0,0,1,1,1,750,1,2,"""CONTINUOUS_TRADING""","""AQEU"""
"""BID""",7.102,750,51442,2023-09-01 07:00:24.633271,"""INSERT""",null,0,null,false,null,0,2,0,8.1,947,947,7.104,750,2257,false,false,51442,0,750,1,1,1,1,750,1,3,"""CONTINUOUS_TRADING""","""AQEU"""
"""BID""",null,0,null,2023-09-01 07:00:24.633286,"""REMOVE""",7.102,750,51442,false,null,0,0,2,8.1,947,947,7.104,1500,2257,false,false,51442,0,1500,2,1,2,0,0,1,3,"""CONTINUOUS_TRADING""","""AQEU"""
"""BID""",7.104,750,51443,2023-09-01 07:00:24.633283,"""INSERT""",null,0,null,false,null,0,1,1,8.1,947,947,7.104,1500,3007,false,true,51443,0,750,1,1,2,2,1500,1,4,"""CONTINUOUS_TRADING""","""AQEU"""


In [278]:
quotes_collapsed.collect().sample(n=10)

original_order_id,venue,insertion_date,removal_date,latest_event_timestamp,number_of_updates,price_at_insertion,size_at_insertion,execution_size,insertion_level,best_bid_price_at_insertion,best_ask_price_at_insertion,lob_updates,market_state_at_removal,order_lifetime,midpoint_at_insertion,abs_distance_to_midpoint,rel_distance_to_midpoint,log_order_lifetime,removal_mechanism
i64,str,datetime[ns],datetime[ns],datetime[ns],u32,f64,i64,i64,i64,f64,f64,list[struct[3]],str,duration[ns],f64,f64,f64,f64,str
1693554663320037916,"""XETR""",2023-09-01 07:51:03.320052620,2023-09-01 07:51:10.721513943,2023-09-01 07:51:10.721513943,0,7.308,303,0,5,7.316,7.322,[],"""CONTINUOUS_TRADING""",7s 401461323ns,7.319,0.011,0.001503,22.724943,"""Cancel"""
1081350376588433179,"""CEUX""",2023-09-01 13:28:31.680299,2023-09-01 13:28:31.680422,2023-09-01 13:28:31.680422,0,7.408,108,0,1,7.398,7.408,[],"""CONTINUOUS_TRADING""",123µs,7.403,0.005,0.000675,11.71994,"""Cancel"""
1693555664189611776,"""XETR""",2023-09-01 08:07:44.189622266,2023-09-01 08:08:20.557883453,2023-09-01 08:08:20.557883453,0,7.276,607,0,4,7.282,7.288,[],"""CONTINUOUS_TRADING""",36s 368261187ns,7.285,0.009,0.001235,24.316962,"""Cancel"""
1081350376588424650,"""CEUX""",2023-09-01 13:27:57.156600,2023-09-01 13:28:02.665545,2023-09-01 13:28:02.665545,0,7.396,318,0,1,7.396,7.4,[],"""CONTINUOUS_TRADING""",5s 508945µs,7.398,0.002,0.00027,22.429639,"""Cancel"""
1693558627572627021,"""XETR""",2023-09-01 08:57:07.572640171,2023-09-01 08:57:07.581695449,2023-09-01 08:57:07.581695449,0,7.284,347,0,2,7.28,7.282,[],"""CONTINUOUS_TRADING""",9055278ns,7.281,0.003,0.000412,16.018858,"""Cancel"""
1693575620035334595,"""XETR""",2023-09-01 13:40:20.035343681,2023-09-01 13:40:21.040998916,2023-09-01 13:40:21.040998916,0,7.402,436,436,1,7.402,7.406,[],"""CONTINUOUS_TRADING""",1s 5655235ns,7.404,0.002,0.00027,20.728905,"""Trade (full)"""
1693567965443824794,"""XETR""",2023-09-01 11:32:45.443833959,2023-09-01 11:32:47.591315210,2023-09-01 11:32:47.591315210,0,7.292,378,0,1,7.292,7.306,[],"""CONTINUOUS_TRADING""",2s 147481251ns,7.299,0.007,0.000959,21.487561,"""Cancel"""
1693566144696625624,"""XETR""",2023-09-01 11:02:24.696633537,2023-09-01 11:18:01.126340543,2023-09-01 11:18:01.126340543,0,7.27,379,0,9,7.286,7.29,[],"""CONTINUOUS_TRADING""",15m 36s 429707006ns,7.288,0.018,0.00247,27.56534,"""Cancel"""
1693568276090111056,"""XETR""",2023-09-01 11:37:56.090120311,2023-09-01 11:37:58.754277736,2023-09-01 11:37:58.754277736,0,7.318,600,0,2,7.308,7.316,[],"""CONTINUOUS_TRADING""",2s 664157425ns,7.312,0.006,0.000821,21.703154,"""Cancel"""


In [279]:
test_ids = [68707, 60135, 1081350376584944624, 1693572441613402348] # [1693551612541526740, 1693551530008208487, 1693551623820260384]

quotes_inc_eu \
    .filter(pl.col("original_order_id").is_in(test_ids)) \
    .sort(by=[pl.col("original_order_id"), pl.col("event_timestamp")]) \
    .collect() \
    .show(limit=10)

quotes_collapsed \
    .filter(pl.col("original_order_id").is_in(test_ids)) \
    .collect() \
    .show(limit=10)

side,price,size,order_id,event_timestamp,lob_action,old_price,old_size,old_order_id,order_executed,execution_price,execution_size,price_level,old_price_level,best_ask_price,best_ask_size,total_ask_size,best_bid_price,best_bid_size,total_bid_size,is_new_best_price,is_new_best_size,original_order_id,trade_id,size_ahead,orders_ahead,best_ask_num_orders,best_bid_num_orders,level_num_orders_total,level_size_total,total_ask_orders,total_bid_orders,market_state,venue
str,f64,i64,i64,datetime[ns],str,f64,i64,i64,bool,f64,i64,i64,i64,f64,i64,i64,f64,i64,i64,bool,bool,i64,i128,i64,i64,i64,i64,i64,i64,i64,i64,str,str
"""BID""",7.114,1,60135,2023-09-01 07:00:30.228330,"""INSERT""",null,0,null,false,null,0,1,0,8.1,947,947,7.114,1,1508,true,true,60135,0,0,0,1,1,1,1,1,3,"""CONTINUOUS_TRADING""","""AQEU"""
"""BID""",null,0,null,2023-09-01 07:00:30.328332,"""REMOVE""",7.114,1,60135,false,null,0,0,2,8.1,947,947,7.116,750,1507,false,false,60135,0,750,1,1,1,0,0,1,2,"""CONTINUOUS_TRADING""","""AQEU"""
"""BID""",7.128,1,68707,2023-09-01 07:00:37.309316,"""INSERT""",null,0,null,false,null,0,1,0,7.18,203,1150,7.128,1,758,true,true,68707,0,0,0,1,1,1,1,2,2,"""CONTINUOUS_TRADING""","""AQEU"""
"""BID""",null,0,null,2023-09-01 07:00:37.310658,"""REMOVE""",7.128,1,68707,true,7.128,1,0,1,7.18,203,1150,5.0,757,757,true,true,68707,551,0,0,1,1,0,0,2,1,"""CONTINUOUS_TRADING""","""AQEU"""
"""ASK""",7.16,108,1081350376584944624,2023-09-01 07:19:55.827492,"""INSERT""",null,0,null,false,null,0,10,0,7.128,650,34042,7.12,493,19640,false,false,1081350376584944624,null,8510,12,1,3,1,108,34,29,"""CONTINUOUS_TRADING""","""CEUX"""
"""ASK""",null,0,null,2023-09-01 07:19:59.266546,"""REMOVE""",7.16,108,1081350376584944624,false,null,0,0,13,7.122,270,35213,7.114,145,18707,false,false,1081350376584944624,null,9789,17,2,1,0,0,38,25,"""CONTINUOUS_TRADING""","""CEUX"""
"""ASK""",7.39,391,1693572441613402348,2023-09-01 12:47:21.613411366,"""INSERT""",null,0,null,false,null,0,1,1,7.39,496,1171621,7.386,3260,673859,false,true,1693572441613402348,null,105,1,2,4,2,496,585,391,"""CONTINUOUS_TRADING""","""XETR"""
"""ASK""",7.39,294,1693572441613402348,2023-09-01 12:47:21.781597887,"""UPDATE""",7.39,391,1693572441613402348,true,7.39,97,1,1,7.39,294,1171419,7.386,3260,671104,false,true,1693572441613402348,1693572441781569214,0,0,1,4,1,294,584,391,"""CONTINUOUS_TRADING""","""XETR"""
"""ASK""",null,0,null,2023-09-01 12:47:21.781629029,"""REMOVE""",7.39,294,1693572441613402348,false,null,0,0,1,7.392,5057,1171125,7.386,3260,671104,true,true,1693572441613402348,null,0,0,4,4,0,0,583,391,"""CONTINUOUS_TRADING""","""XETR"""


original_order_id,venue,insertion_date,removal_date,latest_event_timestamp,number_of_updates,price_at_insertion,size_at_insertion,execution_size,insertion_level,best_bid_price_at_insertion,best_ask_price_at_insertion,lob_updates,market_state_at_removal,order_lifetime,midpoint_at_insertion,abs_distance_to_midpoint,rel_distance_to_midpoint,log_order_lifetime,removal_mechanism
i64,str,datetime[ns],datetime[ns],datetime[ns],u32,f64,i64,i64,i64,f64,f64,list[struct[3]],str,duration[ns],f64,f64,f64,f64,str
60135,"""AQEU""",2023-09-01 07:00:30.228330,2023-09-01 07:00:30.328332,2023-09-01 07:00:30.328332,0,7.114,1,0,1,7.114,8.1,[],"""CONTINUOUS_TRADING""",100002µs,7.607,0.493,0.064809,18.420701,"""Cancel"""
68707,"""AQEU""",2023-09-01 07:00:37.309316,2023-09-01 07:00:37.310658,2023-09-01 07:00:37.310658,0,7.128,1,1,1,7.128,7.18,[],"""CONTINUOUS_TRADING""",1342µs,7.154,0.026,0.003634,14.109672,"""Trade (full)"""
1081350376584944624,"""CEUX""",2023-09-01 07:19:55.827492,2023-09-01 07:19:59.266546,2023-09-01 07:19:59.266546,0,7.16,108,0,10,7.12,7.128,[],"""CONTINUOUS_TRADING""",3s 439054µs,7.124,0.036,0.005053,21.958462,"""Cancel"""
1693572441613402348,"""XETR""",2023-09-01 12:47:21.613411366,2023-09-01 12:47:21.781629029,2023-09-01 12:47:21.781629029,1,7.39,391,97,1,7.386,7.39,"[{2023-09-01 12:47:21.781597887,7.39,294}]","""CONTINUOUS_TRADING""",168217663ns,7.388,0.002,0.000271,18.940769,"""Trade (partial)"""
